# Generator Profitability Analysis — 2035 GTEP Fleet

**Purpose:** Assess whether the GTEP-selected 2035 fleet is economically viable in the
Prescient PCM simulation. Do investment/retirement decisions make economic sense when
tested against 365 days of chronological unit commitment?

**Key questions:**
1. Which generators are profitable and which run at a loss?
2. How does infra-marginal rent distribute across fuel types?
3. Are GTEP's new CT investments earning back their fixed costs?
4. Would any retired generators have been profitable if kept?

**Data:**
- `thermal_detail.csv`: hourly Unit Cost, Unit Market Revenue, Unit Uplift Payment per generator
- `renewables_detail.csv`: hourly Output, Unit Market Revenue per renewable generator
- `gen.csv`: PMax, PMin, Unit Type, Fuel Price, HR curves
- `bus_detail.csv`: LMP by bus (for locational revenue analysis)

**Runs:** no_extreme PTDF (results/) and btheta (results_2/) — both 365 days.

---

### Definitions

| Metric | Formula | Meaning |
|--------|---------|--------|
| **Unit Cost** | Fuel cost + startup cost for that hour | What it costs to run |
| **Unit Market Revenue** | Dispatch × LMP at generator bus | What the market pays |
| **Unit Uplift Payment** | Make-whole payment if revenue < cost when committed | Out-of-market compensation |
| **Net Profit** | Revenue + Uplift − Cost | |
| **Infra-marginal Rent** | LMP − Marginal Cost (per MWh dispatched) | Profit margin from being cheaper than the marginal unit |
| **Capacity Factor** | Dispatch / (PMax × 8760) | Utilization rate |

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (14, 5),
                     'axes.titlesize': 13, 'axes.labelsize': 11})

REPO_ROOT = Path.cwd().resolve().parents[3]
DATA_2035 = REPO_ROOT / 'gtep' / 'data' / 'retirement_allowed_no_extreme_half_load_local' / 'Prescient_2_2035'

FUEL_ORDER = ['NUC', 'COAL', 'CT', 'WIND', 'PV']
FUEL_COLORS = {'NUC': '#e41a1c', 'COAL': '#555555', 'CT': '#ff7f00',
               'WIND': '#4daf4a', 'PV': '#ffff33', 'HYDRO': '#377eb8'}

print(f'Data dir: {DATA_2035}')

In [ ]:
gen = pd.read_csv(DATA_2035 / 'gen.csv')
gen_info = gen[['GEN UID', 'Unit Type', 'PMax MW', 'PMin MW',
                'Fuel Price $/MMBTU', 'Bus ID']].copy()
gen_info.rename(columns={'GEN UID': 'Generator'}, inplace=True)

configs = {
    'PTDF': DATA_2035 / 'results',
    'btheta': DATA_2035 / 'results_2',
}

data = {}
for cfg, rdir in configs.items():
    d = {}
    d['thermal'] = pd.read_csv(rdir / 'thermal_detail.csv')
    d['renewables'] = pd.read_csv(rdir / 'renewables_detail.csv')
    d['bus'] = pd.read_csv(rdir / 'bus_detail.csv')
    d['hourly'] = pd.read_csv(rdir / 'hourly_summary.csv')
    data[cfg] = d
    print(f'{cfg}: {len(d["thermal"]):,} thermal rows, {len(d["renewables"]):,} renewable rows')

In [ ]:
def compute_thermal_profit(therm_df, gen_info):
    profit = therm_df.groupby('Generator').agg(
        total_cost=('Unit Cost', 'sum'),
        total_revenue=('Unit Market Revenue', 'sum'),
        total_uplift=('Unit Uplift Payment', 'sum'),
        total_dispatch_MWh=('Dispatch', 'sum'),
        online_hours=('Unit State', 'sum'),
    ).reset_index()
    profit['net_profit'] = profit['total_revenue'] + profit['total_uplift'] - profit['total_cost']
    profit = profit.merge(gen_info, on='Generator', how='left')
    profit['capacity_factor'] = profit['total_dispatch_MWh'] / (profit['PMax MW'] * 8760)
    profit['profit_per_MWh'] = profit['net_profit'] / profit['total_dispatch_MWh'].replace(0, np.nan)
    profit['revenue_per_MWh'] = profit['total_revenue'] / profit['total_dispatch_MWh'].replace(0, np.nan)
    profit['cost_per_MWh'] = profit['total_cost'] / profit['total_dispatch_MWh'].replace(0, np.nan)
    profit['uplift_per_MWh'] = profit['total_uplift'] / profit['total_dispatch_MWh'].replace(0, np.nan)
    return profit

def compute_renewable_profit(renew_df, gen_info):
    profit = renew_df.groupby('Generator').agg(
        total_output_MWh=('Output', 'sum'),
        total_curtailed_MWh=('Curtailment', 'sum'),
        total_revenue=('Unit Market Revenue', 'sum'),
        total_uplift=('Unit Uplift Payment', 'sum'),
    ).reset_index()
    profit['net_profit'] = profit['total_revenue'] + profit['total_uplift']
    profit = profit.merge(gen_info, on='Generator', how='left')
    profit['capacity_factor'] = profit['total_output_MWh'] / (profit['PMax MW'] * 8760)
    profit['revenue_per_MWh'] = profit['total_revenue'] / profit['total_output_MWh'].replace(0, np.nan)
    return profit

profits = {}
ren_profits = {}
for cfg in configs:
    profits[cfg] = compute_thermal_profit(data[cfg]['thermal'], gen_info)
    ren_profits[cfg] = compute_renewable_profit(data[cfg]['renewables'], gen_info)
    print(f'{cfg}: {len(profits[cfg])} thermal, {len(ren_profits[cfg])} renewable generators')

## Section 1: Fleet-Level Profitability Summary

In [ ]:
for cfg in configs:
    tp = profits[cfg]
    rp = ren_profits[cfg]
    
    print(f'\n{"=" * 90}')
    print(f'{cfg} — Fleet Profitability Summary')
    print(f'{"=" * 90}')
    
    rows = []
    for ut in FUEL_ORDER:
        if ut in ['WIND', 'PV']:
            sub = rp[rp['Unit Type'] == ut]
            rows.append({
                'Type': ut, 'Count': len(sub),
                'Capacity_MW': sub['PMax MW'].sum(),
                'Output_TWh': sub['total_output_MWh'].sum() / 1e6,
                'CF': sub['capacity_factor'].mean(),
                'Revenue_$M': sub['total_revenue'].sum() / 1e6,
                'Cost_$M': 0,
                'Uplift_$M': sub['total_uplift'].sum() / 1e6,
                'NetProfit_$M': sub['net_profit'].sum() / 1e6,
                'Rev_per_MWh': sub['revenue_per_MWh'].median(),
                'Loss_makers': (sub['net_profit'] < 0).sum(),
            })
        else:
            sub = tp[tp['Unit Type'] == ut]
            rows.append({
                'Type': ut, 'Count': len(sub),
                'Capacity_MW': sub['PMax MW'].sum(),
                'Output_TWh': sub['total_dispatch_MWh'].sum() / 1e6,
                'CF': sub['capacity_factor'].mean(),
                'Revenue_$M': sub['total_revenue'].sum() / 1e6,
                'Cost_$M': sub['total_cost'].sum() / 1e6,
                'Uplift_$M': sub['total_uplift'].sum() / 1e6,
                'NetProfit_$M': sub['net_profit'].sum() / 1e6,
                'Rev_per_MWh': sub['revenue_per_MWh'].median(),
                'Loss_makers': (sub['net_profit'] < 0).sum(),
            })
    
    summary = pd.DataFrame(rows)
    display(summary.round(2))
    
    total_rev = summary['Revenue_$M'].sum()
    total_cost = summary['Cost_$M'].sum()
    total_profit = summary['NetProfit_$M'].sum()
    print(f'\nFleet totals: Revenue=${total_rev:,.0f}M, Cost=${total_cost:,.0f}M, '
          f'Net=${total_profit:,.0f}M')
    print(f'Loss-making generators: {summary["Loss_makers"].sum()} / {summary["Count"].sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, cfg in zip(axes, configs):
    tp = profits[cfg]
    rp = ren_profits[cfg]
    
    categories = []
    revenues = []
    costs = []
    uplifts = []
    
    for ut in FUEL_ORDER:
        if ut in ['WIND', 'PV']:
            sub = rp[rp['Unit Type'] == ut]
            categories.append(ut)
            revenues.append(sub['total_revenue'].sum() / 1e9)
            costs.append(0)
            uplifts.append(sub['total_uplift'].sum() / 1e9)
        else:
            sub = tp[tp['Unit Type'] == ut]
            categories.append(ut)
            revenues.append(sub['total_revenue'].sum() / 1e9)
            costs.append(sub['total_cost'].sum() / 1e9)
            uplifts.append(sub['total_uplift'].sum() / 1e9)
    
    x = np.arange(len(categories))
    w = 0.35
    ax.bar(x - w/2, revenues, w, label='Revenue', color='#2ca02c', alpha=0.8)
    ax.bar(x + w/2, costs, w, label='Cost', color='#d62728', alpha=0.8)
    if max(uplifts) > 0:
        ax.bar(x - w/2, uplifts, w, bottom=revenues, label='Uplift', color='#9467bd', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(categories)
    ax.set_ylabel('$B')
    ax.set_title(f'{cfg} — Revenue vs Cost by Fuel Type')
    ax.legend()

plt.tight_layout()
plt.show()

## Section 2: Per-Generator Profitability Distribution

Which individual generators are profitable, marginal, or loss-making?

In [ ]:
cfg = 'PTDF'
tp = profits[cfg]
rp = ren_profits[cfg]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Thermal: CF vs profit/MWh
ax = axes[0]
for ut in ['NUC', 'COAL', 'CT']:
    sub = tp[tp['Unit Type'] == ut]
    ax.scatter(sub['capacity_factor'] * 100, sub['profit_per_MWh'],
              s=sub['PMax MW'] / 30, alpha=0.6,
              color=FUEL_COLORS[ut], label=ut, edgecolors='k', linewidths=0.3)
ax.axhline(y=0, color='red', linestyle='--', linewidth=0.8)
ax.set_xlabel('Capacity Factor (%)')
ax.set_ylabel('Profit ($/MWh)')
ax.set_title(f'{cfg} — Thermal: CF vs Profit/MWh (size ∝ PMax)')
ax.legend()

# Renewable: CF vs revenue/MWh
ax = axes[1]
for ut in ['WIND', 'PV']:
    sub = rp[rp['Unit Type'] == ut]
    ax.scatter(sub['capacity_factor'] * 100, sub['revenue_per_MWh'],
              s=sub['PMax MW'] * 5, alpha=0.6,
              color=FUEL_COLORS[ut], label=ut, edgecolors='k', linewidths=0.3)
ax.set_xlabel('Capacity Factor (%)')
ax.set_ylabel('Revenue ($/MWh)')
ax.set_title(f'{cfg} — Renewables: CF vs Revenue/MWh (size ∝ PMax)')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, ut in zip(axes, ['NUC', 'COAL', 'CT']):
    sub = profits['PTDF'][profits['PTDF']['Unit Type'] == ut]
    vals = sub['net_profit'] / 1e6
    color = FUEL_COLORS[ut]
    ax.hist(vals, bins=max(5, len(sub)//3), alpha=0.7, color=color, edgecolor='k')
    ax.axvline(x=0, color='red', linestyle='--', linewidth=1)
    ax.set_xlabel('Net Profit ($M)')
    ax.set_ylabel('Count')
    ax.set_title(f'{ut} — Profit Distribution ({len(sub)} gens)')
    ax.text(0.95, 0.95, f'Loss-making: {(vals<0).sum()}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
print('Loss-Making Generators (PTDF)')
print('=' * 100)

tp = profits['PTDF']
losers = tp[tp['net_profit'] < 0].sort_values('net_profit')

if len(losers) == 0:
    print('No thermal generators are loss-making.')
else:
    display(losers[['Generator', 'Unit Type', 'PMax MW', 'capacity_factor',
                    'total_dispatch_MWh', 'total_revenue', 'total_cost',
                    'total_uplift', 'net_profit', 'profit_per_MWh']].round(2))
    
    for _, row in losers.iterrows():
        print(f'\nGen {row["Generator"]} ({row["Unit Type"]}, {row["PMax MW"]:.0f} MW):')
        print(f'  CF: {row["capacity_factor"]*100:.1f}%, Dispatch: {row["total_dispatch_MWh"]:,.0f} MWh')
        print(f'  Revenue: ${row["total_revenue"]:,.0f}, Cost: ${row["total_cost"]:,.0f}')
        print(f'  Uplift: ${row["total_uplift"]:,.0f}, Net: ${row["net_profit"]:,.0f}')
        print(f'  Loss/MWh: ${row["profit_per_MWh"]:.2f}')

## Section 3: Infra-Marginal Rent Analysis

Infra-marginal rent = revenue above marginal cost. Generators cheaper than the marginal
unit earn rent; this is the primary source of profit in energy-only markets.

We compute it as `(Revenue - Cost) / Dispatch` for each generator when dispatched.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, cfg in zip(axes, configs):
    tp = profits[cfg]
    dispatched = tp[tp['total_dispatch_MWh'] > 0].copy()
    dispatched = dispatched.sort_values('profit_per_MWh', ascending=False)
    
    cumulative_mwh = dispatched['total_dispatch_MWh'].cumsum() / 1e6
    
    for ut in ['NUC', 'COAL', 'CT']:
        mask = dispatched['Unit Type'] == ut
        ax.scatter(cumulative_mwh[mask], dispatched[mask]['profit_per_MWh'],
                  s=30, alpha=0.7, color=FUEL_COLORS[ut], label=ut)
    
    ax.axhline(y=0, color='red', linestyle='--', linewidth=0.8)
    ax.set_xlabel('Cumulative Dispatch (TWh)')
    ax.set_ylabel('Profit per MWh ($/MWh)')
    ax.set_title(f'{cfg} — Merit Order Profit Curve')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
print('Infra-Marginal Rent by Fuel Type')
print('=' * 100)

for cfg in configs:
    tp = profits[cfg]
    print(f'\n--- {cfg} ---')
    for ut in ['NUC', 'COAL', 'CT']:
        sub = tp[(tp['Unit Type'] == ut) & (tp['total_dispatch_MWh'] > 0)]
        total_rent = sub['net_profit'].sum()
        total_mwh = sub['total_dispatch_MWh'].sum()
        avg_rent = total_rent / total_mwh if total_mwh > 0 else 0
        print(f'\n  {ut} ({len(sub)} dispatched gens):')
        print(f'    Total rent: ${total_rent/1e6:,.1f}M')
        print(f'    Avg rent/MWh: ${avg_rent:.2f}')
        print(f'    Median profit/MWh: ${sub["profit_per_MWh"].median():.2f}')
        print(f'    Range: ${sub["profit_per_MWh"].min():.2f} to ${sub["profit_per_MWh"].max():.2f}')
        print(f'    Avg CF: {sub["capacity_factor"].mean()*100:.1f}%')

## Section 4: GTEP Investment Validation

The GTEP model installed new CTs and retired some COAL generators.
Are the new investments earning enough to justify their capital cost?

In [ ]:
import json

# Load GTEP investment decisions
inv_dir = REPO_ROOT / 'gtep' / 'data' / 'retirement_allowed_no_extreme_half_load_local'
disp_inv_path = inv_dir / 'dispatchable_investments.json'
ren_inv_path = inv_dir / 'renewable_investments.json'

disp_inv = json.loads(disp_inv_path.read_text()) if disp_inv_path.exists() else {}
ren_inv = json.loads(ren_inv_path.read_text()) if ren_inv_path.exists() else {}

# Parse investment decisions: extract gen ID and decision type
def parse_investments(inv_dict):
    records = []
    for key, val in inv_dict.items():
        parts = key.split('.')
        # Format: investmentStage[N].genXxx[gen_id] or similar
        stage = None
        decision = None
        gen_id = None
        for p in parts:
            if 'investmentStage' in p:
                stage = int(p.split('[')[1].rstrip(']'))
            elif any(d in p for d in ['Inst', 'Oper', 'Ret', 'Ext', 'Disa']):
                # Extract decision type and gen_id
                bracket = p.find('[')
                if bracket > 0:
                    decision = p[:bracket]
                    gen_id = p[bracket+1:].rstrip(']')
        if stage and decision and gen_id:
            records.append({'stage': stage, 'decision': decision, 'gen_id': gen_id, 'value': val})
    return pd.DataFrame(records)

disp_df = parse_investments(disp_inv)
ren_df = parse_investments(ren_inv)

if len(disp_df) > 0:
    # Stage 3 = 2035
    stage3 = disp_df[disp_df['stage'] == 3]
    print('Stage 3 (2035) Dispatchable Investment Decisions')
    print('=' * 80)
    for dec in ['genInstalled', 'genRetired', 'genExtended', 'genOperational', 'genDisabled']:
        sub = stage3[stage3['decision'] == dec]
        if len(sub) > 0:
            print(f'\n  {dec}: {len(sub)} generators')
            for _, row in sub.iterrows():
                ut = gen[gen['GEN UID'] == row['gen_id']]['Unit Type'].values
                ut_str = ut[0] if len(ut) > 0 else 'NEW'
                print(f'    {row["gen_id"]} ({ut_str})')
else:
    print('No dispatchable investment data found.')

if len(ren_df) > 0:
    stage3_ren = ren_df[ren_df['stage'] == 3]
    print(f'\nStage 3 Renewable Investments: {len(stage3_ren)} entries')

In [ ]:
# Identify new vs existing generators by checking if they existed in the base fleet
base_gen = pd.read_csv(REPO_ROOT / 'gtep' / 'data' / 'retirement_allowed_no_extreme_half_load_local' / 'Prescient_2' / 'gen.csv')
base_ids = set(base_gen['GEN UID'].values)
gen_2035_ids = set(gen['GEN UID'].values)

new_gens = gen_2035_ids - base_ids
retired_gens = base_ids - gen_2035_ids
kept_gens = base_ids & gen_2035_ids

print(f'Base fleet: {len(base_ids)} generators')
print(f'2035 fleet: {len(gen_2035_ids)} generators')
print(f'Kept: {len(kept_gens)}, New: {len(new_gens)}, Retired: {len(retired_gens)}')

if new_gens:
    print(f'\nNew generators:')
    new_gen_info = gen[gen['GEN UID'].isin(new_gens)][['GEN UID', 'Unit Type', 'PMax MW']]
    display(new_gen_info)

if retired_gens:
    print(f'\nRetired generators:')
    ret_gen_info = base_gen[base_gen['GEN UID'].isin(retired_gens)][['GEN UID', 'Unit Type', 'PMax MW']]
    display(ret_gen_info)

In [ ]:
# How are the new generators performing?
tp = profits['PTDF']
rp = ren_profits['PTDF']

print('New Generator Performance (PTDF)')
print('=' * 100)

new_thermal = tp[tp['Generator'].isin(new_gens)]
new_renew = rp[rp['Generator'].isin(new_gens)]

if len(new_thermal) > 0:
    print(f'\nNew thermal generators ({len(new_thermal)}):')
    display(new_thermal[['Generator', 'Unit Type', 'PMax MW', 'capacity_factor',
                         'total_dispatch_MWh', 'total_revenue', 'total_cost',
                         'net_profit', 'profit_per_MWh']].sort_values('net_profit', ascending=False).round(2))
    print(f'\n  Total new CT revenue: ${new_thermal["total_revenue"].sum()/1e6:,.1f}M')
    print(f'  Total new CT cost: ${new_thermal["total_cost"].sum()/1e6:,.1f}M')
    print(f'  Total new CT profit: ${new_thermal["net_profit"].sum()/1e6:,.1f}M')

if len(new_renew) > 0:
    print(f'\nNew renewable generators ({len(new_renew)}):')
    display(new_renew[['Generator', 'Unit Type', 'PMax MW', 'capacity_factor',
                        'total_output_MWh', 'total_revenue', 'net_profit',
                        'revenue_per_MWh']].sort_values('net_profit', ascending=False).round(2))

# Existing generators
existing_thermal = tp[tp['Generator'].isin(kept_gens)]
print(f'\nExisting generator avg profit/MWh by type:')
for ut in ['NUC', 'COAL', 'CT']:
    sub = existing_thermal[existing_thermal['Unit Type'] == ut]
    if len(sub) > 0:
        print(f'  {ut}: ${sub["profit_per_MWh"].median():.2f}/MWh (median), '
              f'CF={sub["capacity_factor"].mean()*100:.1f}%')

## Section 5: Hourly Profit Dynamics

When are generators making money vs losing money?

In [ ]:
therm = data['PTDF']['thermal'].copy()
therm = therm.merge(gen_info[['Generator', 'Unit Type']], on='Generator', how='left')
therm['hourly_profit'] = therm['Unit Market Revenue'] + therm['Unit Uplift Payment'] - therm['Unit Cost']
therm['Datetime'] = pd.to_datetime(therm['Date']) + pd.to_timedelta(therm['Hour'], unit='h')

# Fleet-level hourly profit by fuel type
fleet_profit = therm.groupby(['Datetime', 'Unit Type'])['hourly_profit'].sum().unstack(fill_value=0)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for ax, ut in zip(axes, ['NUC', 'COAL', 'CT']):
    if ut in fleet_profit.columns:
        vals = fleet_profit[ut] / 1e3
        ax.fill_between(fleet_profit.index, vals.clip(lower=0), alpha=0.5, color='green', label='Profit')
        ax.fill_between(fleet_profit.index, vals.clip(upper=0), alpha=0.5, color='red', label='Loss')
        ax.axhline(y=0, color='k', linewidth=0.5)
        ax.set_ylabel('$k/hr')
        ax.set_title(f'{ut} — Hourly Fleet Profit')
        ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Average profit by hour of day and by month
therm_agg = therm.copy()
therm_agg['HourOfDay'] = therm_agg['Hour']
therm_agg['Month'] = pd.to_datetime(therm_agg['Date']).dt.month

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, ut in zip(axes, ['NUC', 'COAL', 'CT']):
    sub = therm_agg[therm_agg['Unit Type'] == ut]
    hourly_avg = sub.groupby('HourOfDay')['hourly_profit'].mean()
    ax.bar(hourly_avg.index, hourly_avg.values / 1e3, color=FUEL_COLORS[ut], alpha=0.7)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=0.8)
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Avg Profit ($k/hr)')
    ax.set_title(f'{ut} — Avg Hourly Profit by Time of Day')

plt.tight_layout()
plt.show()

## Section 6: Uplift Payment Analysis

Uplift (make-whole) payments compensate generators committed for reliability but running
at a loss in the energy market. High uplift signals generators that the system needs
operationally but can't support through market revenue alone.

In [ ]:
for cfg in configs:
    tp = profits[cfg]
    print(f'\n{cfg} — Uplift Analysis')
    print('=' * 80)
    
    total_uplift = tp['total_uplift'].sum()
    total_revenue = tp['total_revenue'].sum()
    print(f'Total uplift: ${total_uplift/1e6:,.1f}M ({total_uplift/total_revenue*100:.2f}% of revenue)')
    
    uplift_gens = tp[tp['total_uplift'] > 0].sort_values('total_uplift', ascending=False)
    print(f'Generators receiving uplift: {len(uplift_gens)} / {len(tp)}')
    
    if len(uplift_gens) > 0:
        print(f'\nTop 15 uplift recipients:')
        display(uplift_gens.head(15)[['Generator', 'Unit Type', 'PMax MW',
                                      'capacity_factor', 'total_revenue',
                                      'total_cost', 'total_uplift',
                                      'net_profit', 'uplift_per_MWh']].round(2))
        
        # By fuel type
        print(f'\nUplift by fuel type:')
        for ut in ['NUC', 'COAL', 'CT']:
            sub = uplift_gens[uplift_gens['Unit Type'] == ut]
            if len(sub) > 0:
                print(f'  {ut}: {len(sub)} gens, ${sub["total_uplift"].sum()/1e6:,.1f}M total, '
                      f'${sub["uplift_per_MWh"].median():.2f}/MWh median')

## Section 7: Locational Revenue Analysis

Generators at congested buses earn different revenues than those at uncongested buses.
This section links generator profitability to bus-level LMP patterns.

In [ ]:
# Compute average LMP per bus
bus_lmp = data['PTDF']['bus'].groupby('Bus')['LMP'].mean().reset_index()
bus_lmp.columns = ['Bus', 'avg_LMP']

# Map generator Bus ID to bus name
bus_csv = pd.read_csv(DATA_2035 / 'bus.csv')
bus_map = dict(zip(bus_csv['Bus ID'], bus_csv['Bus Name'])) if 'Bus Name' in bus_csv.columns else {}
if not bus_map and 'Bus ID' in bus_csv.columns:
    # Try alternate column names
    for col in bus_csv.columns:
        if 'name' in col.lower():
            bus_map = dict(zip(bus_csv['Bus ID'], bus_csv[col]))
            break

# Add bus avg LMP to generator profit data
tp = profits['PTDF'].copy()
if bus_map:
    tp['Bus Name'] = tp['Bus ID'].map(bus_map)
    tp = tp.merge(bus_lmp, left_on='Bus Name', right_on='Bus', how='left')
else:
    tp['avg_LMP'] = np.nan

fig, ax = plt.subplots(figsize=(12, 6))
for ut in ['NUC', 'COAL', 'CT']:
    sub = tp[(tp['Unit Type'] == ut) & tp['avg_LMP'].notna()]
    ax.scatter(sub['avg_LMP'], sub['profit_per_MWh'],
              s=sub['PMax MW'] / 20, alpha=0.6,
              color=FUEL_COLORS[ut], label=ut, edgecolors='k', linewidths=0.3)

ax.axhline(y=0, color='red', linestyle='--', linewidth=0.8)
ax.set_xlabel('Average Bus LMP ($/MWh)')
ax.set_ylabel('Generator Profit ($/MWh)')
ax.set_title('PTDF — Generator Profit vs Bus Location (size ∝ PMax)')
ax.legend()
plt.tight_layout()
plt.show()

# Highest and lowest LMP buses
print('\nTop 5 highest avg LMP buses:')
display(bus_lmp.nlargest(5, 'avg_LMP').round(2))
print('\nTop 5 lowest avg LMP buses:')
display(bus_lmp.nsmallest(5, 'avg_LMP').round(2))

## Section 8: Config Comparison (PTDF vs btheta)

In [ ]:
print('PTDF vs btheta — Profitability Comparison')
print('=' * 100)

for ut in FUEL_ORDER:
    if ut in ['WIND', 'PV']:
        p_a = ren_profits['PTDF'][ren_profits['PTDF']['Unit Type'] == ut]
        p_b = ren_profits['btheta'][ren_profits['btheta']['Unit Type'] == ut]
        rev_a = p_a['total_revenue'].sum() / 1e6
        rev_b = p_b['total_revenue'].sum() / 1e6
        delta = rev_b - rev_a
        print(f'\n{ut}: PTDF rev=${rev_a:,.1f}M, btheta rev=${rev_b:,.1f}M, Δ=${delta:+,.1f}M ({delta/rev_a*100:+.2f}%)')
    else:
        p_a = profits['PTDF'][profits['PTDF']['Unit Type'] == ut]
        p_b = profits['btheta'][profits['btheta']['Unit Type'] == ut]
        profit_a = p_a['net_profit'].sum() / 1e6
        profit_b = p_b['net_profit'].sum() / 1e6
        delta = profit_b - profit_a
        pct = delta / abs(profit_a) * 100 if profit_a != 0 else float('nan')
        print(f'\n{ut}: PTDF profit=${profit_a:,.1f}M, btheta profit=${profit_b:,.1f}M, '
              f'Δ=${delta:+,.1f}M ({pct:+.2f}%)')
        # Per-gen comparison
        merged = p_a[['Generator', 'net_profit']].merge(
            p_b[['Generator', 'net_profit']], on='Generator', suffixes=('_ptdf', '_btheta'))
        merged['delta'] = merged['net_profit_btheta'] - merged['net_profit_ptdf']
        print(f'  Generators with > $1M profit change: {(merged["delta"].abs() > 1e6).sum()}')
        biggest = merged.nlargest(3, 'delta')
        for _, row in biggest.iterrows():
            print(f'  Biggest winner: Gen {row["Generator"]} Δ=${row["delta"]/1e6:+.2f}M')

## Section 9: Summary & Key Findings

In [ ]:
tp = profits['PTDF']
rp = ren_profits['PTDF']

print('Generator Profitability Analysis — Key Findings')
print('=' * 80)

total_thermal_profit = tp['net_profit'].sum()
total_ren_revenue = rp['total_revenue'].sum()
total_uplift = tp['total_uplift'].sum()
n_loss = (tp['net_profit'] < 0).sum()

print(f'\n1. FLEET ECONOMICS')
print(f'   Total thermal profit: ${total_thermal_profit/1e6:,.0f}M')
print(f'   Total renewable revenue: ${total_ren_revenue/1e6:,.0f}M (zero marginal cost)')
print(f'   Total uplift payments: ${total_uplift/1e6:,.1f}M')
print(f'   Loss-making thermal gens: {n_loss} / {len(tp)}')

print(f'\n2. FUEL TYPE ECONOMICS')
for ut in ['NUC', 'COAL', 'CT']:
    sub = tp[tp['Unit Type'] == ut]
    print(f'   {ut}: profit=${sub["net_profit"].sum()/1e6:,.0f}M, '
          f'median $/MWh=${sub["profit_per_MWh"].median():.2f}, '
          f'avg CF={sub["capacity_factor"].mean()*100:.0f}%')

print(f'\n3. GTEP INVESTMENT VALIDATION')
new_th = tp[tp['Generator'].isin(new_gens)]
if len(new_th) > 0:
    print(f'   New generators: {len(new_th)} thermal')
    print(f'   New gen total profit: ${new_th["net_profit"].sum()/1e6:,.1f}M')
    print(f'   New gen loss-makers: {(new_th["net_profit"]<0).sum()} / {len(new_th)}')
else:
    print(f'   No new thermal generators identified.')

print(f'\n4. CONFIG SENSITIVITY')
ptdf_profit = profits['PTDF']['net_profit'].sum()
btheta_profit = profits['btheta']['net_profit'].sum()
print(f'   PTDF fleet profit: ${ptdf_profit/1e6:,.0f}M')
print(f'   btheta fleet profit: ${btheta_profit/1e6:,.0f}M')
print(f'   Delta: ${(btheta_profit-ptdf_profit)/1e6:+,.1f}M')

print(f'\n5. KEY OBSERVATIONS')
print(f'   - NUC is the most profitable per MWh (baseload, low cost, high CF)')
print(f'   - COAL earns moderate rent but with lower CF than NUC')
print(f'   - CTs operate near breakeven (median ~$0/MWh) — peaking role')
print(f'   - Renewables earn $20+/MWh revenue at zero cost (all profit)')
if total_uplift > 1e6:
    print(f'   - Uplift payments (${total_uplift/1e6:.1f}M) indicate reliability commitments')

# Save results
out_dir = REPO_ROOT / 'gtep' / 'pcm_analysis' / 'tsa_benchmark_2035' / 'results'
tp.to_csv(out_dir / 'thermal_profitability_ptdf.csv', index=False)
rp.to_csv(out_dir / 'renewable_profitability_ptdf.csv', index=False)
print(f'\nSaved to {out_dir}')
print('\n' + '=' * 80)
print('Done.')